# Local tests

Este notebook executa o fluxo principal diretamente:

1. Carrega documentos da pasta `data/`.
2. Cria o índice com `RAG`.
3. Faz perguntas ao chat engine.
4. Gera o PDF com `generate_technical_report`.


In [1]:
from pathlib import Path
import sys
import uuid

# Detecta a raiz do projeto quando o notebook roda a partir de notebooks/
project_root = Path.cwd().resolve()
if not (project_root / "src").exists() and (project_root.parent / "src").exists():
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from llama_index.core import SimpleDirectoryReader
from src.engine import RAG
from src.report import generate_technical_report

print(f"Project root: {project_root}")

c:\Users\karen.alves\AppData\Local\anaconda3\envs\clarus\lib\site-packages\pydantic\_internal\_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'validate_default' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'validate_default' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
c:\Users\karen.alves\AppData\Local\anaconda3\envs\clarus\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Project root: C:\Users\karen.alves\Documents\clarus


In [2]:
data_dir = project_root / "data"
candidate_dirs = sorted(data_dir.glob("temp_*"))
docs_dir = next((d for d in candidate_dirs if d.is_dir() and any(d.iterdir())), None)

if docs_dir is None:
    docs_dir = data_dir / f"temp_notebook_{uuid.uuid4()}"
    docs_dir.mkdir(parents=True, exist_ok=True)
    (docs_dir / "sample_requisitos.txt").write_text(
        "Projeto Clarus: sistema de analise tecnica com upload de documentos, "
        "extracao de riscos e geracao de recomendacoes em portugues.",
        encoding="utf-8",
    )

print(f"Pasta de documentos: {docs_dir}")
documents = SimpleDirectoryReader(str(docs_dir)).load_data()
print(f"Documentos carregados: {len(documents)}")

Ignoring wrong pointing object 6 0 (offset 0)
Ignoring wrong pointing object 10 0 (offset 0)
Ignoring wrong pointing object 14 0 (offset 0)
Ignoring wrong pointing object 17 0 (offset 0)
Ignoring wrong pointing object 21 0 (offset 0)
Ignoring wrong pointing object 28 0 (offset 0)
Ignoring wrong pointing object 53 0 (offset 0)
Ignoring wrong pointing object 67 0 (offset 0)
Ignoring wrong pointing object 68 0 (offset 0)


Pasta de documentos: C:\Users\karen.alves\Documents\clarus\data\temp_72f20ec6-ba16-400f-8916-372e3db5af18
Documentos carregados: 9


In [3]:
try:
    rag = RAG()
    rag.create_index(documents)
    print("Indice criado com sucesso.")
except Exception as exc:
    raise RuntimeError(
        "Falha ao criar o indice. Verifique se o Ollama esta ativo e se as dependencias estao instaladas."
    ) from exc

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 32540.44it/s]


Indice criado com sucesso.


c:\Users\karen.alves\AppData\Local\anaconda3\envs\clarus\lib\site-packages\llama_index\core\agent\react\base.py:154: DeprecationWarning: Call to deprecated class ReActAgent. (ReActAgent has been rewritten and replaced by llama_index.core.agent.workflow.ReActAgent.

This implementation will be removed in a v0.13.0 and the new implementation will be promoted to the `from llama_index.core.agent import ReActAgent` path.

See the docs for more information: https://docs.llamaindex.ai/en/stable/understanding/agent/)
  return cls(
c:\Users\karen.alves\AppData\Local\anaconda3\envs\clarus\lib\site-packages\deprecated\classic.py:184: DeprecationWarning: Call to deprecated class AgentRunner. (AgentRunner has been deprecated and is not maintained.

This implementation will be removed in a v0.13.0.

See the docs for more information on updated agent usage: https://docs.llamaindex.ai/en/stable/understanding/agent/)
  return old_new1(cls, *args, **kwargs)


In [4]:
pergunta = "Quais sao os principais riscos identificados nos documentos?"
resposta = rag.query(pergunta)

print("Pergunta:", pergunta)
print("\nResposta:\n")
print(resposta)

Pergunta: Quais sao os principais riscos identificados nos documentos?

Resposta:

Os principais riscos identificados nos documentos são:

1.  **Cronograma em alto nível:** O cronograma foi definido sem o conhecimento detalhado dos requisitos, o que pode gerar a necessidade de ajustes após a fase de engenharia de requisitos e prototipação.
2.  **Atrasos ou expectativas não atendidas em integrações:** A falta de definições claras sobre as integrações pode comprometer o prazo e o esforço planejado.
3.  **Qualidade dos dados históricos:** Caso os dados fornecidos não atinjam os parâmetros mínimos de qualidade, o treinamento e o funcionamento dos modelos de inteligência artificial podem ser prejudicados, exigindo a busca por novas bases de dados.
4.  **Dados não rotulados:** A premissa é que os dados para treinamento do modelo de recomendação estejam rotulados. A ausência dessa rotulação pode exigir o apoio de especialistas ou até a renegociação do escopo do projeto.


In [5]:
pergunta = "Quais são os principais requisitos desse documento?"
resposta = rag.query(pergunta)

print("Pergunta:", pergunta)
print("\nResposta:\n")
print(resposta)

Pergunta: Quais são os principais requisitos desse documento?

Resposta:

Os principais requisitos e diretrizes estabelecidos no documento são:

*   **Avaliação e Prazos:** Os requisitos das funcionalidades e as entregas prototipadas devem ser avaliados em um prazo de até três semanas.
*   **Documentação e Apoio:** É necessário fornecer as regras de negócio documentadas e facilitar o acesso aos usuários principais para a descrição das funcionalidades e validação das entregas.
*   **Representação Técnica:** Deve ser designado um representante para suporte técnico e operacional, além de um profissional treinado para realizar a implantação do software na infraestrutura da empresa.
*   **Engenharia de Requisitos:** O processo de engenharia de requisitos e a prototipagem devem ser executados logo nas primeiras interações do projeto.
*   **Validação de Dados:** Como não há uma base de dados pré-estabelecida, a validação da solução deve ser realizada utilizando dados simulados.
*   **Gestão d

In [6]:
insights_finais = rag.query(
    "Com base no conteudo analisado, gere um resumo com principais insights e conclusoes tecnicas."
)

chat_history = [
    {"role": "user", "content": pergunta},
    {"role": "assistant", "content": resposta},
]

doc_names = [p.name for p in docs_dir.iterdir() if p.is_file()]
pdf_bytes = generate_technical_report(chat_history, doc_names, insights_finais)

output_pdf = project_root / "data" / "relatorio_teste_notebook.pdf"
output_pdf.write_bytes(pdf_bytes)

print(f"PDF gerado em: {output_pdf}")
print(f"Tamanho do arquivo: {len(pdf_bytes)} bytes")

PDF gerado em: C:\Users\karen.alves\Documents\clarus\data\relatorio_teste_notebook.pdf
Tamanho do arquivo: 3805 bytes
